# Silver PSGC Population
The Silver layer transforms the nested PSGC population payload from `public_health.bronze.geography_bronze` into clean, typed municipal population observations.

Silver grain: one row per psgc_code × population_year × source_version.


1. Parse the nested population JSON

    The PSGC population observations are stored inside the Bronze `populations_json` field. The parent PSGC code from the Bronze geography record is used as the Silver `psgc_code`.

In [ ]:
%sql

CREATE OR REPLACE TEMP VIEW psgc_population_observations AS

SELECT
    g.psgc_code AS psgc_code,
    g.source_version,
    g.batch_id,
    population_item.year AS population_year,
    population_item.population AS population_raw,
    TRY_CAST(
        REGEXP_REPLACE(
            TRIM(population_item.population),
            ',',
            ''
        ) AS BIGINT
    ) AS population

FROM public_health.bronze.geography_bronze g

LATERAL VIEW EXPLODE(
    FROM_JSON(
        g.populations_json,
        'array<struct<id:bigint,population:string,psgc:string,year:int>>'
    )
) exploded AS population_item

WHERE g.populations_json IS NOT NULL;

2. Classify population parsing results

    The raw population value is trimmed and converted to a numeric BIGINT.
    - MISSING: no population value was supplied
    - INVALID: a non-null population value cannot be parsed as a number
    - VALID: population was successfully parsed

In [ ]:
%sql

CREATE OR REPLACE TEMP VIEW psgc_population_parsed AS

SELECT
    psgc_code,
    population_year,
    population,
    population_raw,
    CASE
        WHEN population_raw IS NULL THEN 'MISSING'
        WHEN population IS NULL THEN 'INVALID'
        ELSE 'VALID'
    END AS population_parse_status,
    source_version,
    batch_id

FROM psgc_population_observations;

3. Profile the population observations

    This is an inspection step. It does not modify the data.

    The Bronze QC identified 94 non-null population values that could not be parsed as BIGINT; these records are expected to be excluded from the clean Silver population table.

In [ ]:
%sql

SELECT
    source_version,
    COUNT(*) AS total_observations,
    COUNT_IF(population_parse_status = 'VALID') AS valid_population,
    COUNT_IF(population_parse_status = 'MISSING') AS missing_population,
    COUNT_IF(population_parse_status = 'INVALID') AS invalid_population,
    COUNT(DISTINCT psgc_code) AS distinct_psgc_codes
FROM psgc_population_parsed
GROUP BY source_version
ORDER BY source_version;

4. Check the Silver grain

    The intended Silver grain is:
    `psgc_code × population_year × source_version`
    
    There should be at most one row at this grain.

In [ ]:
%sql

SELECT
    psgc_code,
    population_year,
    source_version,
    COUNT(*) AS row_count
FROM psgc_population_parsed
GROUP BY
    psgc_code,
    population_year,
    source_version
HAVING COUNT(*) > 1
ORDER BY
    row_count DESC,
    psgc_code,
    population_year;

5. Validate population values

    This check shows the range of parsed numeric values and the number of missing or negative values. Missing numeric values are expected when the parse status is MISSING or INVALID.

In [ ]:
%sql

SELECT
    COUNT(*) AS total_observations,
    COUNT_IF(population_parse_status = 'VALID') AS valid_population,
    COUNT_IF(population_parse_status = 'MISSING') AS missing_population,
    COUNT_IF(population_parse_status = 'INVALID') AS invalid_population,
    COUNT_IF(population < 0) AS negative_population,
    MIN(population) AS min_population,
    MAX(population) AS max_population
FROM psgc_population_parsed;

6. Validate accepted parse statuses

    The expected population parsing statuses are VALID, MISSING, and INVALID.

In [ ]:
%sql

SELECT DISTINCT
    population_parse_status
FROM psgc_population_parsed
ORDER BY population_parse_status;

7. Create the PSGC population Silver table

The Silver table keeps all parsed population observations, including records with missing or invalid population values. The original value is retained in `population_raw`, and `population_parse_status` makes the data-quality state visible.

Lineage is preserved through `source_version` and `batch_id`.

In [ ]:
%sql

CREATE OR REPLACE TABLE public_health.silver.population_silver
USING DELTA
COMMENT 'PSGC population Silver layer'
AS

SELECT
    psgc_code,
    population_year,
    population,
    population_raw,
    population_parse_status,
    source_version,
    batch_id,
    CURRENT_TIMESTAMP() AS silver_processed_at
FROM psgc_population_parsed;

8. Validate the created Silver table

In [ ]:
%sql

SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT psgc_code) AS distinct_psgc_codes,
    COUNT(DISTINCT population_year) AS distinct_population_years,
    COUNT(DISTINCT source_version) AS distinct_source_versions,
    COUNT_IF(population_parse_status = 'VALID') AS valid_population,
    COUNT_IF(population_parse_status = 'MISSING') AS missing_population,
    COUNT_IF(population_parse_status = 'INVALID') AS invalid_population
FROM public_health.silver.population_silver;

9. Check the final Silver schema

    Expected columns:

| Column | Purpose |
|---|---|
| `psgc_code` | PSGC geography key |
| `population_year` | Year of the population observation |
| `population` | Cleaned numeric population |
| `population_raw` | Original population value from PSGC |
| `population_parse_status` | `VALID`, `MISSING`, or `INVALID` |
| `source_version` | PSGC source version |
| `batch_id` | Bronze ingestion batch |
| `silver_processed_at` | Silver processing timestamp |

In [ ]:
%sql

DESCRIBE public_health.silver.population_silver;

10. Sample the final Silver data

In [ ]:
%sql

SELECT
    psgc_code,
    population_year,
    population,
    population_raw,
    population_parse_status,
    source_version,
    batch_id,
    silver_processed_at
FROM public_health.silver.population_silver
ORDER BY
    psgc_code,
    population_year
LIMIT 20;